# AAI614: Data Science & its Applications

*Notebook 3.2: Practice with Data Cleaning*

<a href="https://colab.research.google.com/github/harmanani/AAI614/blob/main/Week%203/Notebook3.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

Exercise I. Load the following datafile from GitHub

In [11]:
grads = pd.read_csv("https://raw.githubusercontent.com/harmanani/AAI614/main/Week%203/grads.csv")

In [12]:
grads

,Student Name,Avg Hours Studies per Week,GPA,University,Sense of Humour (0-5),Salary
0,George,20,NaN,NYU,3.0,$40k
1,Jerry,35,3.5,Columbia,5.0,$80k
2,Elaine,55,4.0,Columbia,4.2,$60k
3,Cosmo,5,2.0,City College,2.0,$25k
4,Newman,25,2.8,City College,0.0,$50k
5,Frank,35,3.0,Festivus Uni,NaN,$40k
6,Estelle,100,3.2,Festivus Uni,1.7,$0k
7,Leo,15,2.4,Festivus Uni,0.0,$35k
8,Rachel,50,4.0,Columbia,NaN,$75k


Question 1: Identify all the outliers in the above data.  Justify your answers using objective measures.

In [13]:
import numpy as np

# Clean first — Salary is text ("$40k") and University has stray whitespace
grads['University'] = grads['University'].str.strip()
grads['Salary'] = (grads['Salary'].str.strip()
                   .str.replace('$','',regex=False)
                   .str.replace('k','',regex=False).astype(float))

num = ['Avg Hours Studies per Week','GPA','Sense of Humour (0-5)','Salary']

for c in num:
    s = grads[c].dropna()
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    z = (s - s.mean()) / s.std()
    flagged = grads.loc[s[(s < lo) | (s > hi)].index, 'Student Name'].tolist()
    print(f"{c}")
    print(f"  Q1={q1}, Q3={q3}, IQR={iqr}, bounds=[{lo:.1f}, {hi:.1f}]")
    print(f"  IQR outliers: {flagged if flagged else 'none'}")
    print("  z-scores:", {n: round(v,2) for n,v in zip(grads.loc[s.index,'Student Name'], z)})
    print()

Avg Hours Studies per Week
  Q1=20.0, Q3=50.0, IQR=30.0, bounds=[-25.0, 95.0]
  IQR outliers: ['Estelle']
  z-scores: {'George': -0.63, 'Jerry': -0.1, 'Elaine': 0.61, 'Cosmo': -1.16, 'Newman': -0.45, 'Frank': -0.1, 'Estelle': 2.2, 'Leo': -0.8, 'Rachel': 0.43}

GPA
  Q1=2.6999999999999997, Q3=3.625, IQR=0.9250000000000003, bounds=[1.3, 5.0]
  IQR outliers: none
  z-scores: {'Jerry': 0.54, 'Elaine': 1.24, 'Cosmo': -1.55, 'Newman': -0.44, 'Frank': -0.16, 'Estelle': 0.12, 'Leo': -1.0, 'Rachel': 1.24}

Sense of Humour (0-5)
  Q1=0.85, Q3=3.6, IQR=2.75, bounds=[-3.3, 7.7]
  IQR outliers: none
  z-scores: {'George': 0.38, 'Jerry': 1.41, 'Elaine': 1.0, 'Cosmo': -0.14, 'Newman': -1.18, 'Estelle': -0.3, 'Leo': -1.18}

Salary
  Q1=35.0, Q3=60.0, IQR=25.0, bounds=[-2.5, 97.5]
  IQR outliers: none
  z-scores: {'George': -0.2, 'Jerry': 1.41, 'Elaine': 0.6, 'Cosmo': -0.8, 'Newman': 0.2, 'Frank': -0.2, 'Estelle': -1.81, 'Leo': -0.4, 'Rachel': 1.21}



Question 2: There are various data that are missing.  Fill-in the missing data or delete the rows/columns that you think you should delete.  Justify your answer

In [14]:
print("Missing before:")
print(grads.isna().sum())

# Median imputation for both columns with missing values
grads['GPA'] = grads['GPA'].fillna(grads['GPA'].median())
grads['Sense of Humour (0-5)'] = grads['Sense of Humour (0-5)'].fillna(
    grads['Sense of Humour (0-5)'].median())

print("\nMissing after:")
print(grads.isna().sum())
print()
print(grads)

Missing before:
Student Name                  0
Avg Hours Studies per Week    0
GPA                           1
University                    0
Sense of Humour (0-5)         2
Salary                        0
dtype: int64

Missing after:
Student Name                  0
Avg Hours Studies per Week    0
GPA                           0
University                    0
Sense of Humour (0-5)         0
Salary                        0
dtype: int64

  Student Name  Avg Hours Studies per Week  GPA    University  \
0       George                          20  3.1           NYU   
1        Jerry                          35  3.5      Columbia   
2       Elaine                          55  4.0      Columbia   
3        Cosmo                           5  2.0  City College   
4       Newman                          25  2.8  City College   
5        Frank                          35  3.0  Festivus Uni   
6      Estelle                         100  3.2  Festivus Uni   
7          Leo                       

Question 3: Reload the data and fill-in the data using mean method as well as the frequent method.

In [15]:
# Reload fresh — Q2's median imputation must not carry over
grads = pd.read_csv("https://raw.githubusercontent.com/harmanani/AAI614/main/Week%203/grads.csv")
grads['University'] = grads['University'].str.strip()
grads['Salary'] = (grads['Salary'].str.strip()
                   .str.replace('$','',regex=False)
                   .str.replace('k','',regex=False).astype(float))

num = ['Avg Hours Studies per Week','GPA','Sense of Humour (0-5)','Salary']

# Mean method
mean_filled = grads.copy()
for c in num:
    mean_filled[c] = mean_filled[c].fillna(mean_filled[c].mean())

# Most frequent (mode) method
freq_filled = grads.copy()
for c in freq_filled.columns:
    if freq_filled[c].isna().any():
        freq_filled[c] = freq_filled[c].fillna(freq_filled[c].mode()[0])

print("MEAN method:")
print(mean_filled[['Student Name','GPA','Sense of Humour (0-5)']].round(2))
print("\nFREQUENT method:")
print(freq_filled[['Student Name','GPA','Sense of Humour (0-5)']])

MEAN method:
  Student Name   GPA  Sense of Humour (0-5)
0       George  3.11                   3.00
1        Jerry  3.50                   5.00
2       Elaine  4.00                   4.20
3        Cosmo  2.00                   2.00
4       Newman  2.80                   0.00
5        Frank  3.00                   2.27
6      Estelle  3.20                   1.70
7          Leo  2.40                   0.00
8       Rachel  4.00                   2.27

FREQUENT method:
  Student Name  GPA  Sense of Humour (0-5)
0       George  4.0                    3.0
1        Jerry  3.5                    5.0
2       Elaine  4.0                    4.2
3        Cosmo  2.0                    2.0
4       Newman  2.8                    0.0
5        Frank  3.0                    0.0
6      Estelle  3.2                    1.7
7          Leo  2.4                    0.0
8       Rachel  4.0                    0.0


Exercise II. Run the cell below to create a new dataframe called `df_miss`.  Its first column will contain some missing values.

In [16]:
import pandas as pd
import numpy as np
import random

nrows = 10
ncols = 5

# set a seed for random number generation
np.random.seed(314)
# create an array filled with random data
data = np.array(np.random.rand(nrows, ncols))
# put the data to a pandas dataframe
df_miss = pd.DataFrame(data)
# rename the columns
df_miss.columns = ['col_'+str(ii) for ii in range(ncols)]

# randomly set some values to missing
ix0 = np.random.randint(nrows, size=3)
ix1 = np.random.randint(nrows, size=3)

df_miss['col_0'][ix0] = np.nan
df_miss['col_1'][ix1] = np.nan

print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0       NaN       NaN  0.265048  0.783205  0.918001
1  0.827355       NaN  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3       NaN       NaN  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5       NaN  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


/tmp/ipykernel_399/310065607.py:21: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_miss['col_0'][ix0] = np.nan
/tmp/ipykernel_399/310065607.py:22: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting 

Impute the missing values (NaN) in `col_0` (but not `col_1`) with the median.  Store the values in the dataframe by using the parameter `inplace`.  Print the dataframe.

In [19]:
df_miss.fillna({'col_0': df_miss['col_0'].median()}, inplace=True)
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0  0.677205  0.000000  0.265048  0.783205  0.918001
1  0.827355  0.000000  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3  0.677205  0.000000  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5  0.677205  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


Impute the missing values in `col_1` with value 0.  Store the values in the dataframe by using the parameter `inplace`.  Print the dataframe.

In [18]:
df_miss.fillna({'col_1': 0}, inplace=True)
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0       NaN  0.000000  0.265048  0.783205  0.918001
1  0.827355  0.000000  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3       NaN  0.000000  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5       NaN  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720
